# Synset / object mapping validation

## Loading data

### Load synset tree

In [ ]:
import json, csv
from nltk.corpus import wordnet as wn
import networkx as nx

# Build the legit-synset graph
G = nx.DiGraph()
G.add_nodes_from(x.name() for x in wn.all_synsets())
for parent in wn.all_synsets():
    for child in parent.hyponyms():
        G.add_edge(parent.name(), child.name())

# Add the illegit-synset graph
with open(r"D:\BEHAVIOR-1K\asset_pipeline\metadata\custom_synsets.csv") as f:
    reader = csv.DictReader(f)
    for row in reader:
        child = row["custom_synset"].strip()
        parent = wn.synset(row["hypernyms"].strip()).name()
        assert parent in G.nodes, "Could not find " + parent
        G.add_edge(parent, child)

legit_synsets = set(G.nodes)

In [ ]:
def is_leaf_synset(x):
    return G.out_degree[x] == 0

In [ ]:
def canonicalize(s):
    try:
        return wn.synset(s).name()
    except:
        return s

### Load list of task-required synsets

In [ ]:
with open(r"D:\BEHAVIOR-1K\asset_pipeline\metadata\b200_objects.json", "r") as f:
    obj_mapping = json.load(f)
activities = set(obj_mapping.keys())
activities_list = sorted(activities)
task_required_synsets_by_activity = {
    k: [canonicalize(x) for x in obj_mapping[k] if x != "agent.n.01"]
    for k in activities
}
task_required_synsets = {
    x for objs in task_required_synsets_by_activity.values() for x in objs
}
synset_requiring_tasks = {
    s: [t for t, ss in task_required_synsets_by_activity.items() if s in ss]
    for s in task_required_synsets
}

### Load list of categories matching each synset

In [ ]:
import csv
from collections import defaultdict

# Get the category - synset mapping
pairs = {}
synset_to_cat = defaultdict(list)
with open(
    r"D:\BEHAVIOR-1K\asset_pipeline\metadata\category_mapping.csv", newline=""
) as csvfile:
    reader = csv.DictReader(csvfile)
    for row in reader:
        category = row["category"].strip()
        synset = row["synset"].strip()
        if not synset or not category:
            print(f"Skipping problematic row: {row}")
            continue
        canonical_synset = canonicalize(synset)
        if canonical_synset != synset:
            print(
                f"Non-canonical synset {synset} in category mapping replaced with canonical synset {canonical_synset}"
            )
        synset = canonical_synset
        pairs[category] = synset
        synset_to_cat[synset].append(category)
found_synsets = set(pairs.values())
found_categories = set(pairs.keys())

In [ ]:
def is_synset_available(s):
    child_synsets = [s] + list(nx.descendants(G, s))
    for cs in child_synsets:
        if cs in found_synsets:
            return True

    return False

## Analysis

### Problem 1: Task-required synsets that are not included in the WN hiearchy

In [ ]:
illegal_task_required_synsets = {x for x in task_required_synsets - legit_synsets}
legit_task_required_synsets = task_required_synsets - illegal_task_required_synsets
print(
    f"{len(illegal_task_required_synsets)}/{len(task_required_synsets)} task required synsets missing in hierarchy.\n"
)
print(illegal_task_required_synsets)

### Problem 2: Category-mapped synsets that are not included in the WN hiearchy

In [ ]:
found_invalid_synsets = {
    x
    for x in found_synsets
    if x not in legit_synsets and canonicalize(x) not in legit_synsets
}
print(
    f"{len(found_invalid_synsets)}/{len(found_synsets)} category-mapped synsets are illegal."
)
print("\n".join(found_invalid_synsets))

### Problem 3: Categories are mapped to non-leaf synsets

In [ ]:
nonleaf_cats = {
    cat
    for cat, s in pairs.items()
    if not is_leaf_synset(s) and any(cs in found_synsets for cs in nx.descendants(G, s))
}
leaf_cats = found_categories - nonleaf_cats
print(f"{len(nonleaf_cats)} / {len(pairs)} categories mapped to non-leaf synsets.\n")
print(
    "\n\n".join(
        f"{cat}: {pairs[cat]}. Descendants: {sorted(x for x in nx.descendants(G, pairs[cat]) if x in found_synsets)}"
        for cat in sorted(nonleaf_cats)
    )
)

### Problem 4: Task-required synsets that don't have any categories mapping to them?
Caveat: substances are included too

In [ ]:
# How many of the required synsets exist:
from collections import defaultdict

found_task_required_synsets = set()
for s in legit_task_required_synsets:
    if is_synset_available(s):
        found_task_required_synsets.add(
            s
        )  # Only add the sought-after synset, not children

not_found_task_required_synsets = (
    legit_task_required_synsets - found_task_required_synsets
)
print(
    f"{len(not_found_task_required_synsets)}/{len(legit_task_required_synsets)} legitimate task-required synsets don't have corresponding category entries.\n"
)
print(
    "\n".join(
        f"{s}: [{', '.join(act for act in activities_list if s in task_required_synsets_by_activity[act])}]"
        for s in not_found_task_required_synsets
    )
)

### Problem 5: Task-required synsets map to objects from their descendants too (might have unexpected examples)

In [ ]:
descendant_mapper_count = sum(
    1
    for s in found_task_required_synsets
    if set(synset_to_cat[s])
    != {cat for cs in set(nx.descendants(G, s)) | {s} for cat in synset_to_cat[cs]}
)
print(
    f"{descendant_mapper_count} / {len(found_task_required_synsets)} map to objects from descendants."
)

for s in found_task_required_synsets:
    own_cats = set(synset_to_cat[s])
    subtree = set(nx.descendants(G, s)) | {s}
    desc_cats = {cat for cs in subtree for cat in synset_to_cat[cs]}
    if own_cats == desc_cats:
        continue

    print(f"\nSynset {s} gets the below objects:")
    for cs in sorted(subtree):
        if not synset_to_cat[cs]:
            continue
        print(f"From {cs}:", ", ".join(synset_to_cat[cs]))